In [1]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
import openpyxl
import itertools
from IPython.display import display
import seaborn as sns
import textwrap

In [10]:
df = pd.read_excel("../Interviews_Survey/Survey_results.xlsx")
df = df.dropna(subset=df.columns[1:], how="all").reset_index(drop=True)

print(f"n = {len(df)} valid responses")

n = 31 valid responses


In [11]:
choice_questions = [
    "How frequently do you use the CrowdWater app?",
    "How did you first hear about CrowdWater?",
    "Which country do you currently live in?",
    "Did you ever attend a CrowdWater related event?",
    "Have you recommended the CrowdWater app to anyone?",
    "Do you use the CrowdWater app privately, in an educational context or as part of a (research) project of an organization?",
    "Are there any specific water bodies that you monitor closely?",
    "How many people do you know use the CrowdWater app or used the CrowdWater app at some point?",
    "On a scale from 1 to 10, how much do you like using the CrowdWater app?"
]

custom_orders = {
    "How frequently do you use the CrowdWater app?": [
        "Daily", "Weekly", "Monthly", "Sporadically", "Never"
    ],
    "Do you use the CrowdWater app privately, in an educational context or as part of a (research) project of an organization?": [
        "Privately", "In an educational context", "Part of a (research) project of an organization"
    ],
    "How did you first hear about CrowdWater?": [
        "Internet", "Social Media", "Through other people", "Workshop", "University, School", "Open online course", "Other"
    ],
    "Which country do you currently live in?": [
        "Argentina", "Austria", "Costa Rica", "Cyprus", "Germany", "Switzerland", "UK", "USA"
    ],
    "How many people do you know use the CrowdWater app or used the CrowdWater app at some point?": ["0", "1-2", "3-5", "6-10", ">10"],
    "On a scale from 1 to 10, how much do you like using the CrowdWater app?": ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"],
    "Did you ever attend a CrowdWater related event?": ["Yes", "No"],
    "Have you recommended the CrowdWater app to anyone?": ["Yes", "No"],
    "Are there any specific water bodies that you monitor closely?": ["Yes", "No"],
}

custom_paths = {
    "How frequently do you use the CrowdWater app?": "../Products/Survey/Frequency.jpg",
    "Do you use the CrowdWater app privately, in an educational context or as part of a (research) project of an organization?": "../Products/Survey/Usage_context.jpg",
    "How did you first hear about CrowdWater?": "../Products/Survey/Discovery.jpg",
    "Which country do you currently live in?": "../Products/Survey/Country.jpg",
    "Did you ever attend a CrowdWater related event?": "../Products/Survey/Event.jpg",
    "Have you recommended the CrowdWater app to anyone?": "../Products/Survey/Recommendation.jpg",
    "Are there any specific water bodies that you monitor closely?": "../Products/Survey/Water_bodies.jpg",
    "How many people do you know use the CrowdWater app or used the CrowdWater app at some point?": "../Products/Survey/Other_people.jpg",
    "On a scale from 1 to 10, how much do you like using the CrowdWater app?": "../Products/Survey/Scale.jpg"
}

def wrap_labels(labels, width=15):
    return [textwrap.fill(str(l), width) for l in labels]

def plot_bar_chart(df, column, path, top_n=None, horizontal=False, order=None, label_width=15):
    series = df[column].dropna()

    def clean_value(v):
        if isinstance(v, float) and v.is_integer():
            return str(int(v))
        return str(v)

    series = series.apply(clean_value)

    if order:
        counts = series.value_counts().reindex(order).fillna(0).astype(int)
    else:
        counts = series.value_counts()
        if top_n:
            counts = counts.head(top_n)

    n_total = len(series)
    labels = wrap_labels(counts.index, width=label_width)

    fig, ax = plt.subplots(figsize=(16, 9))

    if horizontal:
        ax.barh(labels[::-1], counts.values[::-1], color="teal")
        ax.set_xlabel("Count", fontsize=17)
        ax.tick_params(axis="x", labelsize=12)
        ax.tick_params(axis="y", labelsize=15)
        ax.grid(axis="x", linestyle="--", alpha=0.5)
        ax.set_axisbelow(True)
    else:
        ax.bar(labels, counts.values, color="teal")
        ax.set_ylabel("Count", fontsize=17)
        ax.tick_params(axis="x", labelsize=15, rotation=0)
        ax.tick_params(axis="y", labelsize=12)
        ax.grid(axis="y", linestyle="--", alpha=0.5)
        ax.set_axisbelow(True)

    ax.set_title(f"{column}\n(n = {n_total})", fontsize=20, wrap=True)

    fig.subplots_adjust(left=0.12, right=0.97, top=0.85, bottom=0.18)

    plt.savefig(path, dpi=300, bbox_inches=None)
    plt.close()

In [12]:
for q in choice_questions:
    plot_bar_chart(df, q, path = custom_paths.get(q), horizontal=(q in ["Which country do you currently live in?", "How did you first hear about CrowdWater?"]), order=custom_orders.get(q))

Stats for scale question

In [9]:
col = "On a scale from 1 to 10, how much do you like using the CrowdWater app?"

scale_data = df[col].dropna().astype(int)

print(f"Mean:   {scale_data.mean():.2f}")
print(f"Median: {scale_data.median():.2f}")
print(f"Std:    {scale_data.std():.2f}")
print(f"n:      {len(scale_data)}")

Mean:   7.13
Median: 8.00
Std:    2.09
n:      31
